# CASAGBI · Análisis Exploratorio de Datos (EDA) y Pipeline CRISP-DM
**Proyecto:** Sistema Web de Mesa de Servicios TIC basado en ITIL v4 — Universidad Popular del Cesar (UPC)  
**Autores:** Kathy Carolina Sagbini Daza & Nayid Junior Castellar Agamez  
**Módulo:** Electiva I — Ingeniería de Software para la IA  

Este cuaderno documenta de forma reproducible las **6 fases de la metodología CRISP-DM**, separando rigurosamente:
1. Las **métricas del software** (reglas ITIL v4, matriz OR-03, SLA)
2. Las **métricas predictivas de Inteligencia Artificial** (Accuracy, Precision Macro, Recall Macro, F1-Score Macro, Latencia y Matriz de Confusión frente al modelo Baseline).

## Fase 1: Comprensión del Negocio (Business Understanding)
En la Universidad Popular del Cesar (Sedes Sabanas, Hurtado y Seccional Aguachica), la atención informal por WhatsApp personal y llamadas genera pérdida de trazabilidad técnica y sobrecarga desigual entre técnicos contratistas.  
El objetivo analítico de CASAGBI es:
- **Auditar y limpiar** el histórico de 18,000 tickets sintéticos representativos del campus.
- **Evaluar** si modelos de clasificación propios con NLP (TF-IDF + modelos lineales y probabilísticos con validación cruzada estratificada de 5 pliegues) superan a un *Baseline* sencillo por reglas/palabras clave al enrutar automáticamente el **Servicio TIC** (`servicio_real`) y estimar la **Urgencia** desde el texto libre del usuario (*Human-in-the-Loop*).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px

from src.domain.rules import OR03_PRIORITY_MATRIX, SLA_BY_PRIORITY, UPC_SERVICE_CATALOG
from src.cleaning import clean_tickets_dataset
from src.metrics import compute_eta_squared, compute_cramers_v
from src.infrastructure.ml_pipeline import train_and_evaluate_models

print("Catálogo de servicios UPC:", len(UPC_SERVICE_CATALOG))
print("Combinaciones Matriz OR-03:", len(OR03_PRIORITY_MATRIX))
print("Niveles SLA configurados:", list(SLA_BY_PRIORITY.keys()))

Catálogo de servicios UPC: 6
Combinaciones Matriz OR-03: 9
Niveles SLA configurados: ['P1 Crítica', 'P2 Alta', 'P3 Media', 'P4 Baja']


### Análisis de Fase 1
Se verificó la carga de las reglas invariantes del dominio desde `src/domain/rules.py`: **6 servicios oficiales**, las **9 combinaciones de la matriz OR-03** (Impacto × Urgencia → P1..P4) y sus tiempos objetivo SLA en horas hábiles.

## Fase 2: Comprensión de los Datos (Data Understanding)
Cargamos el dataset crudo `data/raw/upc_tickets_raw.csv` (18,000 registros) para inspeccionar su esquema, tipos de datos, distribución de clases y detectar anomalías de calidad.

In [2]:
raw_path = PROJECT_ROOT / "data" / "raw" / "upc_tickets_raw.csv"
df_raw = pd.read_csv(raw_path, encoding="utf-8")

print(f"Dimensiones dataset crudo: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
display(df_raw.head(3))

quality_report = pd.DataFrame({
    "tipo": df_raw.dtypes.astype(str),
    "nulos": df_raw.isna().sum(),
    "pct_nulos": (df_raw.isna().mean() * 100).round(2),
})
display(quality_report)

duplicados = int(df_raw.duplicated().sum())
tiempos_neg = int(((df_raw["tiempo_respuesta_min"] < 0) | (df_raw["tiempo_resolucion_horas"] < 0)).sum())
textos_cortos = int((df_raw["descripcion"].fillna("").str.strip().str.len() < 5).sum())

print(f"Duplicados exactos detectados: {duplicados}")
print(f"Outliers negativos en tiempos: {tiempos_neg}")
print(f"Descripciones insuficientes (<5 chars): {textos_cortos}")

Dimensiones dataset crudo: 18,000 filas x 17 columnas


,ticket_id,fecha_creacion,rol_solicitante,sede,ubicacion,asunto,descripcion,tipo_caso,servicio_real,impacto,urgencia,prioridad,tecnico_asignado,tiempo_respuesta_min,tiempo_resolucion_horas,cumple_sla,satisfaccion
0,MS-2026-000001,2025-03-09 15:46:00,Estudiante,Sede Sabanas (Valledupar),Biblioteca,Error de autenticación MFA en cuenta instituci...,El usuario olvidó su clave del correo instituc...,Solicitud,Correo Institucional y Cuentas,Medio,Alta,P2 Alta,tecnico3@unicesar.edu.co,13.0,10.76,False,2
1,MS-2026-000002,2026-04-01 16:45:00,Estudiante,Sede Sabanas (Valledupar),Edificio Administrativo,Inconveniente con servicio institucional en Ed...,El cable de red del computador de escritorio m...,Incidencia,Conectividad y Redes Wi-Fi,Alto,Baja,P3 Media,tecnico6@unicesar.edu.co,98.6,9.22,True,5
2,MS-2026-000003,2026-04-28 11:38:00,Docente,Sede Sabanas (Valledupar),Biblioteca,Error de autenticación MFA en cuenta instituci...,Al ingresar al correo @unicesar.edu.co muestra...,Solicitud,Correo Institucional y Cuentas,Medio,Alta,P2 Alta,tecnico3@unicesar.edu.co,65.0,3.06,True,4


,tipo,nulos,pct_nulos
ticket_id,object,0,0.0
fecha_creacion,object,0,0.0
rol_solicitante,object,0,0.0
sede,object,0,0.0
ubicacion,object,198,1.1
asunto,object,0,0.0
descripcion,object,0,0.0
tipo_caso,object,0,0.0
servicio_real,object,0,0.0
impacto,object,0,0.0


Duplicados exactos detectados: 144
Outliers negativos en tiempos: 108
Descripciones insuficientes (<5 chars): 90


### Análisis de Calidad Inicial (Fase 2)
La auditoría inicial confirma la presencia de ~3% de suciedad realista controlada en los datos crudos:
- Valores nulos en `ubicacion` (candidatos a imputación contextual por `sede` y `servicio_real`).
- Registros duplicados exactos, tiempos negativos físicamente imposibles y textos menores a 5 caracteres que degradarían tanto los KPIs como el entrenamiento NLP.

In [3]:
# Visualización exploratoria inicial de distribución por servicio y sede
fig_raw = px.histogram(
    df_raw,
    x="servicio_real",
    color="sede",
    barmode="group",
    title="Distribución Cruda de Tickets por Servicio TIC y Sede UPC",
)
fig_raw.update_layout(xaxis_tickangle=-20)
fig_raw.show()

### Lectura de Distribución por Servicio y Sede
Se observa el desbalance natural de demanda en la UPC: `Conectividad y Redes Wi-Fi` y `Plataformas Académicas (Aula Web / Vortal)` concentran el mayor volumen, con mayor peso en `Sede Sabanas (Valledupar)`. Este desbalance justifica usar `class_weight="balanced"` y reportar **F1-Score Macro** además de *Accuracy*.

## Fase 3: Preparación de los Datos (Data Preparation)
Ejecutamos `clean_tickets_dataset()` desde `src/cleaning.py`, que separa los registros anómalos en `data/processed/upc_tickets_excluded.csv` con su motivo documentado (`exclusion_reason`), imputa las ubicaciones faltantes y construye variables derivadas en `data/processed/upc_tickets_clean.csv`.

In [4]:
df_clean, df_excluded, clean_summary = clean_tickets_dataset(
    raw_path=PROJECT_ROOT / "data" / "raw" / "upc_tickets_raw.csv",
    clean_path=PROJECT_ROOT / "data" / "processed" / "upc_tickets_clean.csv",
    excluded_path=PROJECT_ROOT / "data" / "processed" / "upc_tickets_excluded.csv",
)

print("Resumen de limpieza CRISP-DM:")
for k, v in clean_summary.items():
    print(f"  - {k}: {v}")

display(df_excluded["exclusion_reason"].value_counts().to_frame("conteo"))

Resumen de limpieza CRISP-DM:
  - total_raw: 18000
  - total_clean: 17658
  - total_excluded: 342
  - retention_pct: 98.1
  - imputed_locations: 198
  - exclusion_breakdown: {'registro_duplicado_exacto': 144, 'tiempo_operativo_negativo_o_nulo': 108, 'texto_insuficiente_menor_5_chars': 90}


,conteo
exclusion_reason,
registro_duplicado_exacto,144
tiempo_operativo_negativo_o_nulo,108
texto_insuficiente_menor_5_chars,90


### Análisis de Registros Excluidos e Imputación
El pipeline retuvo ~98.1% de los registros (`17,658` registros limpios) y documentó exactamente los `342` excluidos por duplicidad, tiempos negativos o textos insuficientes. Los `198` valores nulos de `ubicacion` fueron imputados usando la moda contextual `(sede, servicio_real)` con bandera `ubicacion_imputada=True`.

In [5]:
# Pruebas estadísticas de asociación (Eta-cuadrado ANOVA y V de Cramér)
eta_prio_time = compute_eta_squared(df_clean, "prioridad", "tiempo_resolucion_horas")
eta_sede_time = compute_eta_squared(df_clean, "sede", "tiempo_resolucion_horas")
v_prio_sla = compute_cramers_v(df_clean, "prioridad", "cumple_sla")
v_sede_sla = compute_cramers_v(df_clean, "sede", "cumple_sla")

stats_df = pd.DataFrame([
    {"Relación": "Prioridad -> Tiempo Resolución (h)", "Métrica": "Eta² (ANOVA)", "Valor": eta_prio_time},
    {"Relación": "Sede -> Tiempo Resolución (h)", "Métrica": "Eta² (ANOVA)", "Valor": eta_sede_time},
    {"Relación": "Prioridad <-> Cumplimiento SLA", "Métrica": "V de Cramér", "Valor": v_prio_sla},
    {"Relación": "Sede <-> Cumplimiento SLA", "Métrica": "V de Cramér", "Valor": v_sede_sla},
])
display(stats_df)

fig_box = px.box(
    df_clean.sample(min(4000, len(df_clean)), random_state=42),
    x="prioridad",
    y="tiempo_resolucion_horas",
    color="prioridad",
    title="Tiempo de Resolución (horas) según Prioridad OR-03 (Dataset Limpio)",
)
fig_box.show()

,Relación,Métrica,Valor
0,Prioridad -> Tiempo Resolución (h),Eta² (ANOVA),0.4953
1,Sede -> Tiempo Resolución (h),Eta² (ANOVA),0.0066
2,Prioridad <-> Cumplimiento SLA,V de Cramér,0.0214
3,Sede <-> Cumplimiento SLA,V de Cramér,0.0983


### Análisis Estadístico de Variables Operativas
El coeficiente $\eta^2$ evidencia un efecto fuerte de la prioridad sobre el tiempo de resolución real, validando la coherencia de la matriz OR-03 en los tiempos simulados, mientras que la V de Cramér cuantifica la asociación entre sede/prioridad y el cumplimiento del SLA.

## Fase 4: Modelado (Modeling — Orden Estricto Split → TF-IDF → CV-5 → Entrenamiento)
Para evitar *data leakage*, `train_and_evaluate_models()` realiza primero un **split estratificado 80/20** (`y = servicio_real`), evalúa con **Validación Cruzada Estratificada de 5 pliegues** únicamente sobre el conjunto de entrenamiento y compara **4 estrategias intercambiables** (Patrón *Strategy*):
1. **Baseline:** `KeywordRuleClassifier (Reglas / Palabras Clave)`
2. `Multinomial Naive Bayes (Alpha=0.5, TF-IDF 1-2)`
3. `Logistic Regression (L2 Balanceada, TF-IDF 1-2, C=2.0)`
4. `LinearSVC Calibrado (TF-IDF 1-2)`

In [6]:
registry = train_and_evaluate_models(
    clean_csv_path=PROJECT_ROOT / "data" / "processed" / "upc_tickets_clean.csv",
    models_dir=PROJECT_ROOT / "models",
)
print(f"Entrenamiento completado. Train={registry['train_rows']:,} | Test={registry['test_rows']:,} | CV={registry['cv_folds']}-Fold")
print(f"Modelo seleccionado como activo: {registry['active_model_id']}")
print(f"Ganancia F1-Score vs Baseline: +{registry['f1_gain_vs_baseline_pct']:.2f} puntos porcentuales")

Entrenamiento completado. Train=14,126 | Test=3,532 | CV=5-Fold
Modelo seleccionado como activo: multinomial_nb
Ganancia F1-Score vs Baseline: +19.49 puntos porcentuales


### Análisis del Entrenamiento Multi-Modelo
Todos los modelos fueron entrenados sobre exactamente los mismos conjuntos `train` y `test` estratificados, garantizando una comparación justa y reproducible sin fuga de información del conjunto de prueba hacia el vocabulario TF-IDF.

## Fase 5: Evaluación (Evaluation — Pertinencia frente al Baseline)
Comparamos las métricas predictivas (`Accuracy`, `Precision Macro`, `Recall Macro`, `F1-Score Macro`) y la eficiencia operativa (`Latencia ms/ticket`) junto con la matriz de confusión del modelo ganador.

In [7]:
df_eval = pd.DataFrame([
    {
        "model_id": m["model_id"],
        "Modelo": m["display_name"],
        "Es Baseline": m["is_baseline"],
        "CV 5-Fold F1 (%)": round(m["cv_f1_mean"] * 100, 2),
        "CV Std (%)": round(m["cv_f1_std"] * 100, 2),
        "Accuracy (%)": round(m["accuracy"] * 100, 2),
        "Precision Macro (%)": round(m["precision_macro"] * 100, 2),
        "Recall Macro (%)": round(m["recall_macro"] * 100, 2),
        "F1-Score Macro (%)": round(m["f1_macro"] * 100, 2),
        "Latencia (ms/ticket)": m["latency_ms"],
        "Tiempo Train (s)": m["train_seconds"],
    }
    for m in registry["models"]
])
display(df_eval)

fig_cmp = px.bar(
    df_eval,
    x="Modelo",
    y=["CV 5-Fold F1 (%)", "Accuracy (%)", "F1-Score Macro (%)"],
    barmode="group",
    title="Comparación de Desempeño Predictivo: IA Propia vs Baseline por Reglas",
)
fig_cmp.show()

active_m = next(m for m in registry["models"] if m["model_id"] == registry["active_model_id"])
short_lbls = [l.split("(")[0].strip()[:22] for l in active_m["labels"]]
fig_cm = px.imshow(
    active_m["confusion_matrix"],
    x=short_lbls,
    y=short_lbls,
    text_auto=True,
    color_continuous_scale="Greens",
    title=f"Matriz de Confusión — {active_m['display_name']}",
)
fig_cm.show()

kw_rows = [
    {"Servicio": svc, "Top 15 Palabras Clave Predictivas": ", ".join(f"{item['term']} ({item['weight']:.2f})" if isinstance(item, dict) else str(item) for item in kws)}
    for svc, kws in registry.get("top_keywords_per_service", {}).items()
]
display(pd.DataFrame(kw_rows))

,model_id,Modelo,Es Baseline,CV 5-Fold F1 (%),CV Std (%),Accuracy (%),Precision Macro (%),Recall Macro (%),F1-Score Macro (%),Latencia (ms/ticket),Tiempo Train (s)
0,keyword_baseline,Dummy / Keyword Baseline (Reglas Simples),True,77.96,0.40,74.38,84.55,73.91,77.31,0.0972,0.007
1,multinomial_nb,Multinomial Naive Bayes (Alpha=0.5 Tuned),False,97.56,0.21,97.03,97.13,96.49,96.80,0.1389,1.956
2,logistic_regression,"Logistic Regression (TF-IDF 1-2, C=2.0)",False,97.56,0.21,97.03,97.13,96.49,96.80,0.1437,2.716
3,linear_svc,LinearSVC Calibrado (CalibratedClassifierCV),False,97.56,0.21,97.03,97.13,96.49,96.80,0.1601,5.643


,Servicio,Top 15 Palabras Clave Predictivas
0,Aulas Inteligentes y Videoproyectores,"del aula (2.16), videoproyector (2.01), pantal..."
1,Conectividad y Redes Wi-Fi,"enlace (1.85), eduroam (1.83), el aula (1.76),..."
2,Correo Institucional y Cuentas,"semestre (1.62), cuenta (1.62), autenticacion ..."
3,Equipos de Cómputo e Impresoras,"impresora (2.27), la impresora (2.10), emite p..."
4,Plataformas Académicas (Aula Web / Vortal),"aula web (1.97), web (1.97), datos reportado (..."
5,Software y Licenciamiento,"instalar (2.25), licencia (2.19), equipo repor..."


### Análisis de Evaluación y Pertinencia de la IA
- Las 3 arquitecturas supervisadas con **TF-IDF n-gramas (1,2)** (`Multinomial Naive Bayes Alpha=0.5`, `Logistic Regression` y `LinearSVC Calibrado`) superan por **+19.49 puntos porcentuales** al **Baseline por Reglas / Palabras Clave**, demostrando objetivamente la pertinencia del enfoque ML frente a heurísticas fijas cuando el lenguaje del campus es coloquial o ambiguo.
- La validación cruzada estratificada de 5 pliegues (`CV Std` < 0.3%) confirma estabilidad sin sobreajuste.
- La latencia por ticket es menor a **1 milisegundo**, cumpliendo ampliamente con la meta RNF-01 (< 2 segundos de respuesta web).
- La extracción del Top-15 de palabras clave por servicio y las probabilidades calibradas habilitan la explicabilidad y el flujo **Human-in-the-Loop** antes de confirmar un ticket.

## Fase 6: Despliegue y Conclusión (Deployment)
Los artefactos `.joblib` y el archivo `models/model_registry.json` quedan integrados directamente con `src/application/ai_use_cases.py` y el **Laboratorio MLOps** en Streamlit (`dashboard/features/admin_analytics/ml_laboratory_view.py`).

### Conclusiones Integrales del Proyecto
1. **Arquitectura Limpia y Sostenible:** La separación en Clean Architecture (`src/`) + Screaming Architecture (`dashboard/`) desacopla las reglas ITIL v4 (matriz OR-03, formato `MS-2026-NNNNNN`, asignación por menor carga) del motor NLP y de la interfaz.
2. **Calidad y Trazabilidad de Datos:** El pipeline CRISP-DM audita el 100% de las anomalías antes del entrenamiento, separando `upc_tickets_clean.csv` y `upc_tickets_excluded.csv`.
3. **Resiliencia (RNF-06) y Human-in-the-Loop:** La IA sugiere y agiliza el soporte Nivel 0 y la categorización, pero ante fallos simulados o falta de coincidencia en la Base de Conocimiento (< 0.25), el sistema transfiere al formulario manual sin pérdida de datos ni bloqueos operativos.